In [1]:
import geopandas as gpd
import pandas as pd
import requests
import zipfile, io
import tempfile
import re
import fnmatch
import os

In [2]:
paths = pd.read_csv('pluto_zips.csv')

In [5]:
url1 = paths.full_path[0]

In [15]:
url1

'https://www1.nyc.gov/assets/planning/download/zip/data-maps/open-data/mappluto_02b.zip'

In [16]:
def unzip_to_temp(zip_path):
    r = requests.get(zip_path)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    tmp = tempfile.TemporaryDirectory()
    z.extractall(tmp.name)
    
    return(tmp)
    

In [17]:
tmp_path = unzip_to_temp(url1)

In [19]:
tmp_path.name

'/var/folders/_3/1tvp9zss7kjg9tz64yv51pz80000gn/T/tmp6dwp8vf6'

In [22]:
matches = []
for root, dirnames, filenames in os.walk(tmp_path.name):
    for filename in fnmatch.filter(filenames, '*.shp'):
        matches.append(os.path.join(root, filename))

In [23]:
matches

['/var/folders/_3/1tvp9zss7kjg9tz64yv51pz80000gn/T/tmp6dwp8vf6/MapPLUTO_02B/Bronx/bxmappluto.shp',
 '/var/folders/_3/1tvp9zss7kjg9tz64yv51pz80000gn/T/tmp6dwp8vf6/MapPLUTO_02B/Staten_Island/simappluto.shp',
 '/var/folders/_3/1tvp9zss7kjg9tz64yv51pz80000gn/T/tmp6dwp8vf6/MapPLUTO_02B/Manhattan/mnmappluto.shp',
 '/var/folders/_3/1tvp9zss7kjg9tz64yv51pz80000gn/T/tmp6dwp8vf6/MapPLUTO_02B/Brooklyn/bkmappluto.shp',
 '/var/folders/_3/1tvp9zss7kjg9tz64yv51pz80000gn/T/tmp6dwp8vf6/MapPLUTO_02B/Queens/qnmappluto.shp']

In [24]:
def get_shp_path(dire):

    matches = []
    for root, dirnames, filenames in os.walk(dire.name):
        for filename in fnmatch.filter(filenames, '*.shp'):
            matches.append(os.path.join(root, filename))
    return(matches)

In [26]:
matches = get_shp_path(dire = tmp_path)

In [28]:
def negative_filter(filter_word, list_to_filter):
    
    r = (f'(?i)^((?!{filter_word}).)*$')
    reg = re.compile(r)
    
    return_list = []
    
    for s in list_to_filter:
        if reg.match(s):
            return_list.append(s)
    return(return_list)

In [29]:
files = negative_filter(filter_word = 'unclipped', list_to_filter = matches)

In [32]:
files

['/var/folders/_3/1tvp9zss7kjg9tz64yv51pz80000gn/T/tmp6dwp8vf6/MapPLUTO_02B/Bronx/bxmappluto.shp',
 '/var/folders/_3/1tvp9zss7kjg9tz64yv51pz80000gn/T/tmp6dwp8vf6/MapPLUTO_02B/Staten_Island/simappluto.shp',
 '/var/folders/_3/1tvp9zss7kjg9tz64yv51pz80000gn/T/tmp6dwp8vf6/MapPLUTO_02B/Manhattan/mnmappluto.shp',
 '/var/folders/_3/1tvp9zss7kjg9tz64yv51pz80000gn/T/tmp6dwp8vf6/MapPLUTO_02B/Brooklyn/bkmappluto.shp',
 '/var/folders/_3/1tvp9zss7kjg9tz64yv51pz80000gn/T/tmp6dwp8vf6/MapPLUTO_02B/Queens/qnmappluto.shp']

In [48]:
def append_shapefiles(shapefile_list):
    
    length = len(shapefile_list)
    
    if length == 1:
        return gpd.read_file(shapefile_list[0])
    else:
        g = gpd.read_file(shapefile_list[0])
        for x in range(1, length):
            g = g.append(gpd.read_file(shapefile_list[x]))
    return(g)

In [49]:
g = append_shapefiles(files)

In [50]:
g

,borough,block,lot,cd2,ct2000,cb2000,schoolDist,ccDist,zipCode,fireComp,...,FAR,maxAllwFAR,boroCode,TBL,tract2000,xCoord,yCoord,cogisPluto,geometry,boro
0,BX,2260.0,1,201,17,1005,7,8,10454,L029,...,0.73,2.0,2.0,22600001.0,0017,1006036.0,232287.0,1,"POLYGON ((1006037.583 232232.963, 1005989.348 ...",NaN
1,BX,2260.0,4,201,17,1005,7,8,10454,L029,...,0.02,2.0,2.0,22600004.0,0017,1006090.0,232258.0,1,"POLYGON ((1006158.949 232165.434, 1006110.092 ...",NaN
2,BX,2260.0,10,201,17,1005,7,8,10454,L029,...,1.13,2.0,2.0,22600010.0,0017,1006170.0,232214.0,1,"POLYGON ((1006312.367 232078.922, 1006264.443 ...",NaN
3,BX,2260.0,17,201,17,1005,7,8,10454,L029,...,4.11,2.0,2.0,22600017.0,0017,1006330.0,232125.0,1,"POLYGON ((1006334.152 232067.097, 1006322.325 ...",NaN
4,BX,2260.0,18,201,17,1005,7,8,10454,L029,...,4.58,2.0,2.0,22600018.0,0017,1006357.0,232111.0,1,"POLYGON ((1006389.543 232006.103, 1006322.325 ...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319838,QN,16241.0,200,414,None,None,0,0,00000,None,...,0.00,0.5,4.0,162410200.0,0000,0.0,0.0,4,None,NaN
319839,QN,16287.0,1,484,None,None,0,0,00000,None,...,0.00,0.5,4.0,162870001.0,0000,0.0,0.0,2,None,NaN
319840,QN,16287.0,150,414,None,None,0,0,00000,None,...,0.00,0.5,4.0,162870150.0,0000,0.0,0.0,4,None,NaN
319841,QN,16340.0,10,484,918,None,27,32,11697,E329,...,0.00,0.9,4.0,163400010.0,0918,0.0,0.0,4,None,NaN
